# Faster R-CNN Experiments: GOST Stamp Detection

**Цель:** Обучить Faster R-CNN для детекции штампов на строительных чертежах.

**Данные:** 500 synthetic (train) + 49 real (val)

**Метрики:** IoU, Precision, Recall, F1 на 49 реальных изображениях

**Подход:** Single training run, ResNet50 FPN backbone, 30 epochs, GPU T4 (Colab)

**Resize:** Компромисс — synthetic (200 DPI) ресайзится по PPI (100), real — fallback на 800px.


## 1. Colab Setup

⚠️ **Запустить только один раз!** Клонирует репозиторий (sparse checkout) и устанавливает зависимости. Генерирует 500 синтетических изображений.

In [1]:
import sys
import random
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    PROJECT_DIR = Path.cwd()
    sys.path.insert(0, str(PROJECT_DIR / "src"))
    %cd /content
    !rm -rf aie-group-2-sapar
    !git init aie-group-2-sapar
    %cd aie-group-2-sapar
    !git sparse-checkout set project
    !git remote add origin https://github.com/Sapar-hub/aie-group-2-sapar.git
    !git pull origin main
    %cd project
    !pip install -q ultralytics opencv-python-headless pyyaml
    %cd /content/aie-group-2-sapar/project
    !nvidia-smi
    !python scripts/generate_synthetic.py --output data/ --num-gost 250 --num-copy 250 --dpi 200
else:
    PROJECT_DIR = Path.cwd().parent
    sys.path.insert(0, str(PROJECT_DIR / "src"))

## 2. Imports & Data Setup

In [ ]:
import torch
import torchvision
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.transforms import functional as F
import numpy as np
import matplotlib.pyplot as plt
import cv2
from torch.utils.data import Dataset, DataLoader, Subset

from evaluation.metrics import DetectionResult, bbox_iou, yolo_to_pixel, compute_metrics, print_metrics
from data.loader import load_image_and_labels

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
torch.cuda.manual_seed_all(RANDOM_STATE)

DATA_DIR = PROJECT_DIR / "data"
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
ARTIFACTS_DIR.mkdir(exist_ok=True)
(ARTIFACTS_DIR / "models").mkdir(exist_ok=True)
(ARTIFACTS_DIR / "metrics").mkdir(exist_ok=True)
(ARTIFACTS_DIR / "figures").mkdir(exist_ok=True)

IMAGE_TEST_DIR = DATA_DIR / "images" / "test"
LABEL_TEST_DIR = DATA_DIR / "labels" / "test"
IMAGE_TRAIN_DIR = DATA_DIR / "images" / "train"
LABEL_TRAIN_DIR = DATA_DIR / "labels" / "train"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
print(f"Train images: {len(list(IMAGE_TRAIN_DIR.glob('*.png')) + list(IMAGE_TRAIN_DIR.glob('*.jpg')))}")
print(f"Test images: {len(list(IMAGE_TEST_DIR.glob('*.png')) + list(IMAGE_TEST_DIR.glob('*.jpg')))}")

## 3. Dataset & DataLoader

**Компромисс PPI:**
- Synthetic (train): DPI=200 известен → ресайз по целевому PPI (100).
  Штамп 185×55mm всегда ~728×216px — единый физический масштаб.
- Real (test): PPI неизвестен → fallback на MAX_SIZE=800 по длинной стороне.
- Bbox корректируются пропорционально scale.
- Цель: synthetic учится на физических размерах, real не ломается.

In [ ]:
# ─── Compromise: PPI-based resize for synthetic (known DPI),
#     pixel-based fallback for real (unknown PPI).
# Synthetic: generated at 200 DPI → resample to TARGET_PPI so
#   stamp ~185mm всегда имеет одинаковый физический масштаб.
# Real: PPI неизвестен, fallback на MAX_SIZE.
TARGET_PPI = 100
SYNTH_DPI = 200
MAX_SIZE = 800

def resize_ppi(image, boxes, src_ppi, target_ppi=TARGET_PPI):
    """Resize based on physical PPI ratio. Preserves real-world scale."""
    scale = target_ppi / src_ppi
    if scale >= 1.0:
        return image, boxes, 1.0
    new_w = int(image.shape[1] * scale)
    new_h = int(image.shape[0] * scale)
    image = cv2.resize(image, (new_w, new_h), interpolation=cv2.INTER_AREA)
    if len(boxes) > 0:
        boxes = boxes * scale
    return image, boxes, scale

def resize_maxsize(image, boxes, max_size=MAX_SIZE):
    """Fallback: resize so longest side <= max_size."""
    h, w = image.shape[:2]
    scale = max_size / max(h, w)
    if scale >= 1.0:
        return image, boxes, 1.0
    new_w, new_h = int(w * scale), int(h * scale)
    image = cv2.resize(image, (new_w, new_h), interpolation=cv2.INTER_AREA)
    if len(boxes) > 0:
        boxes = boxes * scale
    return image, boxes, scale

class GOSTDataset(Dataset):
    def __init__(self, image_dir, label_dir, synth_dpi=None):
        self.image_dir = Path(image_dir)
        self.label_dir = Path(label_dir)
        self.synth_dpi = synth_dpi
        self.image_paths = sorted(list(self.image_dir.glob("*.png")) + list(self.image_dir.glob("*.jpg")))

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        img = cv2.imread(str(img_path))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]

        label_files = list(self.label_dir.glob(f"*-{img_path.stem}.txt"))
        if label_files:
            with open(label_files[0]) as f:
                labels = [list(map(float, line.strip().split())) for line in f if line.strip()]
            boxes = []
            for label in labels:
                _, cx, cy, bw, bh = label
                x1 = (cx - bw / 2) * w
                y1 = (cy - bh / 2) * h
                x2 = (cx + bw / 2) * w
                y2 = (cy + bh / 2) * h
                boxes.append([x1, y1, x2, y2])
            boxes = torch.tensor(boxes, dtype=torch.float32)
        else:
            boxes = torch.zeros((0, 4), dtype=torch.float32)

        if self.synth_dpi is not None:
            img, boxes, _ = resize_ppi(img, boxes, self.synth_dpi)
        else:
            img, boxes, _ = resize_maxsize(img, boxes)
        img_tensor = F.to_tensor(img)

        labels_tensor = torch.ones(len(boxes), dtype=torch.int64) if len(boxes) > 0 else torch.zeros(0, dtype=torch.int64)
        target = {"boxes": boxes, "labels": labels_tensor}
        return img_tensor, target

train_dataset = GOSTDataset(IMAGE_TRAIN_DIR, LABEL_TRAIN_DIR, synth_dpi=SYNTH_DPI)
test_dataset = GOSTDataset(IMAGE_TEST_DIR, LABEL_TEST_DIR)
print(f"Train: {len(train_dataset)}, Test: {len(test_dataset)}")

## 4. Train/Test Split (80/20)

Логический split для synthetic (500) и real (49):
- Synthetic: 400 train + 100 val для early stopping
- Real: 39 val (используются RCNN как val) + 10 holdout для финальной оценки
Устраняет data leakage и позволяет отслеживать overfit.
YOLO `gost_stamp.yaml` продолжает использовать все 49 как val.

In [ ]:
# ─── Synthetic 80/20 split (400 train + 100 val)
n_synth = len(train_dataset)
synth_indices = list(range(n_synth))
random.Random(RANDOM_STATE).shuffle(synth_indices)
n_val_synth = int(0.2 * n_synth)
val_synth_idx = synth_indices[:n_val_synth]
train_synth_idx = synth_indices[n_val_synth:]
print(f"Synthetic: {len(train_synth_idx)} train + {len(val_synth_idx)} val")

# ─── Real 80/20 split (39 val + 10 holdout test)
all_real_images = sorted(IMAGE_TEST_DIR.glob("*.png")) + sorted(IMAGE_TEST_DIR.glob("*.jpg"))
random.Random(RANDOM_STATE).shuffle(all_real_images)
n_val_real = int(0.8 * len(all_real_images))
val_images = all_real_images[:n_val_real]
test_images = all_real_images[n_val_real:]
print(f"Real: {len(val_images)} val + {len(test_images)} holdout test")
for p in test_images:
    print(f"  Holdout: {p.name}")

## 5. Training

Оптимизатор: Adam с дифференциальным LR (backbone `1e-4`, head `1e-3`).
Train/val split synthetic 400/100. Early stopping patience=5 по val loss. NaN/Inf check.

In [ ]:
import time
start = time.time()

NUM_CLASSES = 2
NUM_EPOCHS = 30
BATCH_SIZE = 8
LR_HEAD = 1e-3
LR_BACKBONE = 1e-4

model = fasterrcnn_resnet50_fpn(weights=None)
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = torchvision.models.detection.faster_rcnn.FastRCNNPredictor(in_features, NUM_CLASSES)
model.to(DEVICE)

# Differential LR
backbone_params = []
head_params = []
for name, p in model.named_parameters():
    if 'box_predictor' in name:
        head_params.append(p)
    else:
        backbone_params.append(p)

optimizer = torch.optim.Adam([
    {'params': backbone_params, 'lr': LR_BACKBONE},
    {'params': head_params, 'lr': LR_HEAD},
], weight_decay=1e-4)
lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)

train_subset = Subset(train_dataset, train_synth_idx)
val_subset = Subset(train_dataset, val_synth_idx)
train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, collate_fn=lambda x: tuple(zip(*x)), num_workers=0, pin_memory=True)
val_loader = DataLoader(val_subset, batch_size=BATCH_SIZE, collate_fn=lambda x: tuple(zip(*x)), num_workers=0, pin_memory=True)

best_val_loss = float('inf')
patience = 5
wait = 0
best_epoch = 0
train_loss_history = []
val_loss_history = []

for epoch in range(NUM_EPOCHS):
    # Train
    model.train()
    epoch_loss = 0
    for images, targets in train_loader:
        images = [img.to(DEVICE) for img in images]
        targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]
        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())

        if not torch.isfinite(losses):
            print(f"NaN/Inf at epoch {epoch+1}, stopping")
            break

        optimizer.zero_grad()
        losses.backward()
        optimizer.step()
        epoch_loss += losses.item()
    lr_scheduler.step()
    avg_train_loss = epoch_loss / len(train_loader)
    train_loss_history.append(avg_train_loss)

    # Validate
    model.eval()
    with torch.no_grad():
        val_loss = 0
        for images, targets in val_loader:
            images = [img.to(DEVICE) for img in images]
            targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]
            loss_dict = model(images, targets)
            val_loss += sum(loss for loss in loss_dict.values()).item()
    avg_val_loss = val_loss / len(val_loader)
    val_loss_history.append(avg_val_loss)

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch {epoch+1}/{NUM_EPOCHS}, Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}")

    # Early stopping
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        best_epoch = epoch
        wait = 0
        torch.save(model.state_dict(), str(ARTIFACTS_DIR / "models" / "rcnn_best.pth"))
    else:
        wait += 1
        if wait >= patience:
            print(f"Early stopping at epoch {epoch+1}, best epoch {best_epoch+1}, best val loss {best_val_loss:.4f}")
            break

elapsed = time.time() - start
print(f"\nTraining time: {elapsed/60:.1f} minutes")

print(f"Weights saved to {ARTIFACTS_DIR / 'models' / 'rcnn_best.pth'}")
loss_history = train_loss_history
val_loss_history_print = val_loss_history

## 6. Loss Plot

Train + Val loss кривые. Early stopping отсекает лишние эпохи.

In [ ]:
plt.figure(figsize=(10, 5))
best_epoch_display = best_epoch + 1
plt.plot(range(1, len(train_loss_history)+1), train_loss_history, marker="o", label="Train Loss")
plt.plot(range(1, len(val_loss_history)+1), val_loss_history, marker="s", label="Val Loss")
plt.axvline(x=best_epoch_display, color="r", linestyle="--", alpha=0.5, label=f"Best epoch {best_epoch_display}")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Faster R-CNN Training & Validation Loss")
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig(ARTIFACTS_DIR / "figures" / "rcnn_loss.png", dpi=150)
plt.show()

## 7. Evaluation on 10 Holdout Real Images

80/20 split real images → 39 YOLO-val + **10 holdout** для честной итоговой оценки.
Threshold sweep `[0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.5]` + greedy matching (max IoU per GT).

In [ ]:
THRESHOLDS = [0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.5]
best_f1 = 0.0
best_conf = 0.0
best_metrics = None
best_results = None

model.eval()

for conf_thresh in THRESHOLDS:
    results_list = []
    for img_path in test_images:
        img, labels = load_image_and_labels(img_path, LABEL_TEST_DIR)
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]

        gt_bbox = yolo_to_pixel(tuple(labels[0]), w, h) if len(labels) > 0 else None

        gt_boxes = np.array([[gt_bbox[0], gt_bbox[1], gt_bbox[0]+gt_bbox[2], gt_bbox[1]+gt_bbox[3]]], dtype=np.float32) if gt_bbox else np.zeros((0, 4), dtype=np.float32)
        img_rgb, boxes_resized, _ = resize_maxsize(img_rgb, gt_boxes)
        if len(boxes_resized) > 0:
            x1, y1, x2, y2 = map(int, boxes_resized[0])
            gt_bbox = [x1, y1, x2 - x1, y2 - y1]

        img_tensor = F.to_tensor(img_rgb).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            preds = model(img_tensor)[0]

        boxes = preds["boxes"].cpu().numpy()
        scores = preds["scores"].cpu().numpy()

        # Greedy matching: find best pred for this GT
        gt_box = np.array([gt_bbox], dtype=np.float32) if gt_bbox else np.zeros((0, 4), dtype=np.float32)
        pred_bbox = None
        best_iou = 0.0
        for j in range(len(boxes)):
            if scores[j] < conf_thresh:
                continue
            pbox = boxes[j]
            iou_val = bbox_iou(
                (int(pbox[0]), int(pbox[1]), int(pbox[2]-pbox[0]), int(pbox[3]-pbox[1])),
                gt_bbox
            ) if gt_bbox else 0.0
            if iou_val > best_iou:
                best_iou = iou_val
                pred_bbox = (int(pbox[0]), int(pbox[1]), int(pbox[2]-pbox[0]), int(pbox[3]-pbox[1]))

        results_list.append(DetectionResult(
            image_name=img_path.name,
            gt_bbox=tuple(gt_bbox) if gt_bbox else None,
            pred_bbox=pred_bbox,
            iou=best_iou,
            found=pred_bbox is not None
        ))

    metrics = compute_metrics(results_list, iou_threshold=0.5)
    f1 = metrics['f1']
    print(f"conf={conf_thresh:.2f} | P={metrics['precision']:.3f} R={metrics['recall']:.3f} F1={f1:.3f} IoU={metrics['iou_mean']:.3f}")
    if f1 > best_f1:
        best_f1 = f1
        best_conf = conf_thresh
        best_metrics = metrics
        best_results = results_list

results_list = best_results
metrics = best_metrics
print(f"\nBest threshold: conf={best_conf}")
print_metrics(metrics, prefix="RCNN ")

## 8. Visualization

Worst/Best по IoU среди 10 holdout.

In [ ]:
sorted_results = sorted(results_list, key=lambda r: r.iou)
worst = sorted_results[0]
best = sorted_results[-1]

fig, axes = plt.subplots(1, 2, figsize=(16, 8))
for ax, result, title_prefix in zip(axes, [worst, best], ["Worst", "Best"]):
    img_path = [p for p in test_images if p.name == result.image_name][0]
    img, _ = load_image_and_labels(img_path, LABEL_TEST_DIR)
    vis = img.copy()
    if result.gt_bbox:
        x, y, bw, bh = result.gt_bbox
        cv2.rectangle(vis, (x, y), (x+bw, y+bh), (0, 255, 0), 3)
    if result.pred_bbox:
        x, y, bw, bh = result.pred_bbox
        cv2.rectangle(vis, (x, y), (x+bw, y+bh), (0, 0, 255), 2)
    ax.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
    ax.set_title(f"{title_prefix} IoU={result.iou:.3f}")
    ax.axis("off")

plt.suptitle("Green=GT, Red=Pred (Faster R-CNN)")
plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / "figures" / "rcnn_best_worst.png", dpi=150)
plt.show()

## 9. Conclusions

Итог: Faster R-CNN на holdout (10 real). Differential LR, Adam, early stopping.

In [ ]:
summary = {
    "model": "Faster R-CNN (ResNet50 FPN)",
    "optimizer": "Adam",
    "lr_backbone": LR_BACKBONE,
    "lr_head": LR_HEAD,
    "data": "400 synthetic train + 100 synthetic val + 39 real (YOLO-val) + 10 real holdout (eval)",
    "best_epoch": best_epoch_display,
    "epochs": len(train_loss_history),
    "train_time_min": round(elapsed/60, 1),
    "best_conf": best_conf,
    "iou_mean": round(metrics['iou_mean'], 3),
    "iou_std": round(metrics['iou_std'], 3),
    "precision": round(metrics['precision'], 3),
    "recall": round(metrics['recall'], 3),
    "f1": round(metrics['f1'], 3),
    "detection_rate": round(metrics['detection_rate'], 3),
}

import json
with open(ARTIFACTS_DIR / "metrics" / "rcnn_results.json", "w") as f:
    json.dump(summary, f, indent=2)

print("Summary saved to artifacts/metrics/rcnn_results.json")
print(json.dumps(summary, indent=2))

## 10. Save Results to Git

⚠️ **Запустить после обучения!** Сохраняет метрики и графики в репозиторий.


In [ ]:
if IN_COLAB:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
    
    %cd /content/aie-group-2-sapar                                                                    
                                                                                                       
      # Configure git                                                                                  
    !git config user.email "183649607+Sapar-hub@users.noreply.github.com"                             
    !git config user.name "Saparmyrat"                                                                
    !git remote set-url origin https://{token}@github.com/Sapar-hub/aie-group-2-sapar.git                                                      
    !git add -A     
    
    # Commit and push
    !git commit -m "exp04: YOLOv8 results and metrics"
    !git push origin main